# 05 — Baseline de modelado

Baseline para predecir `rhythm_label` a partir de las features generadas en `04_windowing_and_feature_engineering.ipynb`.

**Alcance de este notebook**

- Validar mecánica del pipeline: `SimpleImputer → StandardScaler → Clasificador`.
- Split estricto por `case_id` (`GroupKFold` / `GroupShuffleSplit`).
- Manejo de desbalance vía `class_weight="balanced"`.
- Reportar métricas macro, reporte por clase con soporte y matriz de confusión en conteos absolutos con totales por fila y columna.

**Restricciones obligatorias**

- `beat_type` **NO** se usa como predictor (lo verifica `assert_no_forbidden_features`).
- Split por `case_id`, nunca por ventana ni latido aleatorio.
- No reportar cifras antes de ejecutar las celdas con datos reales.

**Pre-requisito**

Haber ejecutado:
1. `03_ecg_loading_and_visualization.ipynb` para descargar los ECG de todos los `case_id` que se quieran usar.
2. `04_windowing_and_feature_engineering.ipynb` para generar `data/processed/features_baseline.parquet` con esos casos.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.modeling import (
    assert_no_forbidden_features,
    build_logreg_pipeline,
    build_rf_pipeline,
    make_group_kfold,
    make_group_split,
    safe_n_splits,
)
from src.evaluation import (
    class_support_per_split,
    classes_missing_in_train,
    compute_macro_metrics,
    confusion_matrix_with_totals,
    per_class_report,
)
from src.utils import get_logger, set_seed

set_seed(config.RANDOM_SEED)
logger = get_logger("nb05")
sns.set_theme(context="notebook", style="whitegrid")

## 2. Carga de la tabla de features

In [ ]:
features_path = config.PROCESSED_DIR / "features_baseline.parquet"
if not features_path.exists():
    raise FileNotFoundError(
        f"No existe {features_path}. Ejecuta primero 04_windowing_and_feature_engineering.ipynb."
    )

df = pd.read_parquet(features_path)

n_before = len(df)

# 1) Eliminar filas con label NaN.
df = df.dropna(subset=[config.TARGET_COLUMN])

# 2) Defensivo: parquets generados con versiones anteriores podían guardar la
#    etiqueta como la cadena literal "nan"/"NaN" en lugar de un NaN real.
mask_string_nan = df[config.TARGET_COLUMN].astype(str).str.strip().str.lower().isin({"nan", "none", ""})
df = df.loc[~mask_string_nan].copy()

n_after = len(df)
logger.info("Filas con label válido: %d / %d", n_after, n_before)

print("Shape:", df.shape)
print("Columnas:", list(df.columns))
df.head()

## 3. Diagnóstico de clases por `case_id` (crítico antes del split)

El split se hace por `case_id`. Si una clase aparece en pocos casos (en el extremo: un único caso), entonces ese caso debe estar en train **o** en test, no en ambos. Si está en test, el modelo nunca verá esa clase en entrenamiento y su recall será 0 por construcción.

Esta celda muestra cuántos latidos por clase tiene cada caso y en cuántos casos distintos aparece cada clase. Si alguna clase aparece en **un único caso**, hay que tomar decisiones explícitas (excluirla, descargar más casos que la contengan, o asumir que su métrica será dura).

In [ ]:
crosstab = (
    df.groupby([config.CASE_ID_COLUMN, config.TARGET_COLUMN])
      .size()
      .unstack(fill_value=0)
)
print("Casos únicos:", crosstab.shape[0])
print("Clases únicas:", crosstab.shape[1])
crosstab

In [ ]:
cases_per_class = (crosstab > 0).sum(axis=0).rename("cases_with_class").sort_values()
totals_per_class = crosstab.sum(axis=0).rename("total_windows").reindex(cases_per_class.index)
summary = pd.concat([totals_per_class, cases_per_class], axis=1)
summary

In [ ]:
singleton_classes = cases_per_class[cases_per_class <= 1].index.tolist()
if singleton_classes:
    print(
        "AVISO: las siguientes clases aparecen en <=1 case_id.\n"
        "Con split por case_id su recall puede ser 0 cuando ese caso queda en test.\n",
        singleton_classes,
    )
else:
    print("Todas las clases aparecen en al menos 2 case_id.")

## 4. Preparación de X, y, groups

Se excluyen como features: `case_id`, `rhythm_label`, `beat_type`, `bad_signal_quality`, e identificadores de ventana (`beat_index`, `start_sample`, `end_sample`).

In [ ]:
non_feature_cols = set(config.FORBIDDEN_FEATURE_COLUMNS) | {
    "beat_index",
    "start_sample",
    "end_sample",
}
feature_cols = [c for c in df.columns if c not in non_feature_cols]

# Bloqueo metodológico: aborta si beat_type u otra columna prohibida se filtra como feature.
assert_no_forbidden_features(feature_cols)

X = df[feature_cols].to_numpy()
y = df[config.TARGET_COLUMN].to_numpy()
groups = df[config.CASE_ID_COLUMN].to_numpy()

print("Features usadas:", feature_cols)
print("X shape:", X.shape, "| y shape:", y.shape, "| grupos únicos:", np.unique(groups).shape[0])
print("NaN en X:", int(np.isnan(X).sum()))  # serán imputados por el Pipeline

## 5. Split por `case_id` y diagnóstico de soporte por split

Se hace un único split (`GroupShuffleSplit`, `test_size=0.2`). Antes de entrenar, se reporta cuántas muestras de cada clase quedaron en train y test.

In [ ]:
train_idx, test_idx = make_group_split(X, y, groups, test_size=0.2, random_state=config.RANDOM_SEED)
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

# Hard-check: ningún case_id puede aparecer en ambos lados.
assert set(groups_train).isdisjoint(set(groups_test)), "Fuga de grupo entre train y test."

print("Casos en train:", sorted(set(groups_train.tolist())))
print("Casos en test :", sorted(set(groups_test.tolist())))
print("Ventanas train:", X_train.shape[0], "| Ventanas test:", X_test.shape[0])

In [ ]:
support_df = class_support_per_split(y_train, y_test)
support_df

In [ ]:
missing = classes_missing_in_train(y_train, y_test)
if missing:
    print(
        "AVISO: estas clases aparecen en test pero NO en train.\n"
        "Por construcción su recall será 0 y arrastrarán f1_macro hacia abajo:\n",
        missing,
    )
else:
    print("Todas las clases de test están representadas en train.")

## 6. Definición del Pipeline

Pipeline scikit-learn: `SimpleImputer(strategy="median") → StandardScaler → Clasificador`. Todas las etapas viven dentro del mismo `Pipeline` para que `fit` se haga solo sobre train (sin fuga). El imputer permite que el pipeline tolere NaN en features sin necesidad de limpiar a mano.

Ambos clasificadores se configuran con `class_weight="balanced"` para penalizar más los errores en clases minoritarias.

In [ ]:
pipelines = {
    "logreg": build_logreg_pipeline(class_weight="balanced"),
    "random_forest": build_rf_pipeline(class_weight="balanced"),
}
for name, pipe in pipelines.items():
    print(f"--- {name} ---")
    print(pipe)

## 7. Entrenamiento, predicción y métricas macro

In [ ]:
results = {}
for name, pipe in pipelines.items():
    logger.info("Entrenando %s...", name)
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    results[name] = {
        "pipeline": pipe,
        "y_pred": y_pred,
        "metrics": compute_macro_metrics(y_test, y_pred),
    }

metrics_df = pd.DataFrame({name: info["metrics"] for name, info in results.items()}).T
metrics_df

## 8. Reporte por clase (con `support`)

La columna `support` indica cuántas muestras reales tiene cada clase en `y_test`. Si una clase tiene `support=0` o muy bajo, su F1 no es interpretable como métrica estable; conviene leerla junto con la matriz de confusión absoluta de la sección siguiente.

In [ ]:
for name, info in results.items():
    print(f"=== {name} — reporte por clase ===")
    rep = per_class_report(y_test, info["y_pred"])
    print(rep.round(3).to_string())
    print()

## 9. Matriz de confusión en conteos absolutos + totales

Se reporta la matriz **sin normalizar** porque con clases muy desbalanceadas los porcentajes pueden ser engañosos.

- Filas = clases reales (`y_true`).
- Columnas = clases predichas (`y_pred`).
- Columna extra `support_true`: cuántas muestras de cada clase real había en test.
- Fila extra `predicted_total`: cuántas predicciones cayeron en cada clase.

In [ ]:
for name, info in results.items():
    cm_df = confusion_matrix_with_totals(y_test, info["y_pred"])
    print(f"=== {name} — matriz de confusión absoluta ===")
    print(cm_df.to_string())

    # Heatmap solo de la submatriz (sin la fila/col de totales).
    inner = cm_df.iloc[:-1, :-1]
    fig, ax = plt.subplots(figsize=(1.0 + 0.9 * inner.shape[1], 1.0 + 0.7 * inner.shape[0]))
    sns.heatmap(
        inner,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=ax,
    )
    ax.set_title(f"Matriz de confusión absoluta — {name}")
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
    plt.tight_layout()
    plt.show()
    print()

## 10. Validación cruzada por grupos (`GroupKFold`)

`safe_n_splits` recorta automáticamente el número de folds al número de `case_id` únicos cuando son menos de los pedidos (`GroupKFold` exige `n_splits <= n_groups`).

Se reporta una métrica por fold y los grupos que cayeron en test de cada fold.

In [ ]:
n_splits_eff = safe_n_splits(config.DEFAULT_N_SPLITS, groups)
print(f"n_splits efectivo (recortado por # de grupos): {n_splits_eff}")

fold_metrics = []
for i, (tr, te) in enumerate(make_group_kfold(X, y, groups, n_splits=n_splits_eff)):
    pipe = build_logreg_pipeline(class_weight="balanced")
    pipe.fit(X[tr], y[tr])
    y_pred = pipe.predict(X[te])
    m = compute_macro_metrics(y[te], y_pred)
    m["fold"] = i
    m["test_groups"] = sorted(set(groups[te].tolist()))
    m["n_test"] = int(len(te))
    fold_metrics.append(m)

cv_df = pd.DataFrame(fold_metrics).set_index("fold")
cv_df

In [ ]:
numeric_cols = [c for c in cv_df.columns if pd.api.types.is_numeric_dtype(cv_df[c])]
cv_df[numeric_cols].agg(["mean", "std", "min", "max"]).round(3)

## 11. Lectura del baseline y próximos pasos

Cómo interpretar lo que ves arriba antes de pasar a búsqueda de hiperparámetros:

1. Si en la sección 3 alguna clase aparece en **≤ 1 case_id**, sus métricas en cualquier fold donde ese caso quede en test no serán informativas. Más casos en `03` mitigan esto.
2. Si en la sección 5 hay clases listadas como *missing in train*, su recall será 0 por construcción y `f1_macro` no es comparable contra runs sin ese problema.
3. La matriz de confusión absoluta de la sección 9 muestra dónde están concentrados los errores. Mirar primero las clases con mayor `support_true` (los errores ahí mueven la métrica) y luego las minoritarias.
4. La tabla de la sección 10 expone la variabilidad entre folds. Una media alta con desviación enorme suele indicar que el resultado depende del fold concreto, no del modelo.

Antes de iterar en modelado:

- Asegurar que `03_ecg_loading_and_visualization.ipynb` haya descargado suficientes `case_id`.
- Re-ejecutar `04_windowing_and_feature_engineering.ipynb` para regenerar `features_baseline.parquet` con todos los casos.
- Volver a correr este notebook entero y comparar la sección 10 contra el run anterior.